# CP5403 Prac 8: Retrieval-Augmented Generation (RAG)

**Reading:** Alammar & Grootendorst, *Hands-On Large Language Models*, Chapter 8 — *Semantic Search and Retrieval-Augmented Generation*.

This prac follows the Week 8 lecture:

- **8.1 Introduction to Retrieval-Augmented Generation** — why we need RAG (hallucinations), what RAG is, and how it works. You will build a working RAG pipeline: retrieve → grounded prompt → generate.
- **8.2 Advanced RAG techniques** — query rewriting, query expansion (multi-query), multi-hop RAG, query routing, reranking, summary (context compression), and agentic RAG.
- **8.3 RAG evaluation** — human vs. LLM evaluation: fluency, perceived utility, citation recall, citation precision.

We finish by scaling the generator up (does a bigger model fix the failures we find along the way?) and wrapping the pipeline in the Gradio chatbot you built in Prac 1 — turning it into a "chat with your data" app.

## Google Colab runtime

This prac is intended to be run using Google Colab. **Make sure to switch to the T4 GPU runtime before running any code — in the top right, next to `Connect`, click the down arrow and then choose `Change runtime type`. Under `Hardware accelerator` select `T4 GPU`, then click save.** The GPU is needed for the larger model in Part 4.

## Learning outcomes

By the end of this prac you should be able to:

- Explain **why** RAG is needed: what hallucination is and how retrieval reduces it.
- Describe **how a basic RAG pipeline works**: a search step followed by a grounded generation step.
- Assemble a RAG pipeline in code: retrieve relevant documents, build a grounded prompt, and generate an answer with an LLM.
- Describe and (for some) implement the **advanced RAG techniques**: query rewriting, query expansion, multi-hop, query routing, reranking, and agentic RAG.
- Explain the four axes used to **evaluate RAG systems** — fluency, perceived utility, citation recall, citation precision — and how LLM-as-a-judge automates them.
- Observe how **model scale** affects RAG quality: faithfulness, instruction-following, and judge reliability.
- Deploy the pipeline as a simple **Gradio chat app**.

Cells marked **Exercise** are for you to complete. Cells marked **Question** should be answered in a markdown cell (they are typical exam/quiz material).

## Part 0 — Setup

Install the libraries we need:

| Library | Used for |
|---|---|
| `sentence-transformers` | embedding model for the retrieval step, and the reranker |
| `faiss-cpu` | fast nearest-neighbor search index |
| `transformers` + `accelerate` | the LLMs for the generation step (as in Prac 1) |
| `gradio` | the chat web app (as in Prac 1) |

In [ ]:
%pip install -q sentence-transformers faiss-cpu transformers accelerate pandas ipywidgets gradio

In [ ]:
import faiss
import numpy as np
import pandas as pd

### The text archive

We use the first section of the Wikipedia article on the film *Interstellar* as our "knowledge base" — the external data source our RAG system will be grounded on. We split it into one sentence per chunk so each chunk can be retrieved independently.

In [ ]:
text = """
Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind.
Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007.
Caltech theoretical physicist and 2017 Nobel laureate in Physics Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm.
Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects.
Interstellar premiered on October 26, 2014, in Los Angeles.
In the United States, it was first released on film stock, expanding to venues using digital projectors.
The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014.
It received acclaim for its performances, direction, screenplay, musical score, visual effects, ambition, themes, and emotional weight.
It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics.
Since its premiere, Interstellar gained a cult following, and now is regarded by many sci-fi experts as one of the best science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy Awards, winning Best Visual Effects, and received numerous other accolades.
"""

# Split into a list of sentence chunks
texts = text.split(".")

# Clean up whitespace and drop empty chunks
texts = [t.strip(" \n") for t in texts if t.strip(" \n")]

print(f"{len(texts)} chunks")
for i, t in enumerate(texts):
    print(f"[{i:2d}] {t[:80]}{'...' if len(t) > 80 else ''}")

## Part 1 (8.1) — Introduction to Retrieval-Augmented Generation

### Why do we need RAG?

> *"Hallucination refers to the generation of content by an LLM that is fluent and syntactically correct but factually inaccurate or unsupported by external evidence."* — Alansari & Luqman (2025)

LLMs answer fluently and confidently — but not always correctly. The leading method the industry turned to remedy this behavior is **RAG**, described in the paper *"Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks"* (Lewis et al., 2020).

### What is RAG?

RAG systems are **text generation systems that incorporate search capabilities** to reduce hallucinations, increase factuality, and/or ground the generation model on a specific dataset. They enable "chat with my data" use cases — grounding an LLM on internal company data or a data source of interest — and power generative search engines (Perplexity, Bing AI, Google Gemini).

### How does RAG work?

A basic RAG pipeline is made up of **two steps**:

```
question ──► 1. SEARCH: retrieve relevant documents ──► 2. GROUNDED GENERATION: LLM prompted with [question + retrieved documents] ──► answer
```

The generation step is called **grounded generation** because the retrieved information establishes a context that grounds the LLM in the data we care about.

Let's build each step in turn.

### Step 1: the retrieval step

The search step works with **embeddings**: every chunk of our archive is turned into a vector, and chunks whose vectors are **nearest** to the query's vector are the most relevant. We find the most relevant information to an input prompt by comparing the similarities between embeddings.

We use a small open-source embedding model, and FAISS as the index that stores the vectors and retrieves nearest neighbors quickly:

In [ ]:
from sentence_transformers import SentenceTransformer

# A small, strong open-source embedding model
embedding_model = SentenceTransformer("thenlper/gte-small")

embeds = np.array(embedding_model.encode(texts, show_progress_bar=True))
print(embeds.shape)

In [ ]:
# Build the search index
dim = embeds.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(np.float32(embeds))
print("Vectors in index:", index.ntotal)

In [ ]:
def search(query, number_of_results=3):
    # 1. Embed the query into the same space as the documents
    query_embed = embedding_model.encode([query])

    # 2. Retrieve the nearest neighbors
    distances, similar_item_ids = index.search(
        np.float32(query_embed), number_of_results
    )

    # 3. Format the results
    texts_np = np.array(texts)
    results = pd.DataFrame(
        data={
            "texts": texts_np[similar_item_ids[0]],
            "distance": distances[0],
        }
    )

    print(f"Query: '{query}'\nNearest neighbors:")
    return results

In [ ]:
results = search("how precise was the science")
results

The top result (smallest distance) answers the question even though it shares no keywords with the query — retrieval matched on *meaning*.

### Step 2: grounded generation

Now we add an LLM to the end of the pipeline. We present the question **and** the top retrieved documents to the LLM, and ask it to answer using the context provided.

We'll start with a deliberately *small* instruction-tuned model (0.5B parameters) so downloads and generation are fast — and so its weaknesses are easy to see. In Part 4 we'll swap in a model 8× larger and compare. *(The chapter's hosted example uses Cohere's `co.chat` with a `documents` argument — which also returns exact span-level citations. Small local models can't reliably cite spans, but the flow is the same.)*

In [ ]:
from transformers import pipeline
from transformers.utils import logging as hf_logging

# Keep the notebook output clean (hide the library's info/deprecation warnings)
hf_logging.set_verbosity_error()

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    dtype="auto",
    device_map="auto",
)

### The RAG prompt

The prompt template is the central place where we hand the retrieved documents to the LLM: relevant information first, then the question, with an instruction to answer *from the provided information*.

Note the `do_sample=False` in the generation call below — recall temperature and sampling from Prac 1. Here we disable sampling entirely (greedy decoding, always the most probable next token) so that answers are reproducible and differences come from the *retrieval*, not from random sampling.

In [ ]:
PROMPT_TEMPLATE = """Relevant information:
{context}

Provide a concise answer the following question using the relevant information provided above:
{question}"""


def rag_answer(question, number_of_results=3, verbose=True):
    # ----- 1. Retrieval -----
    results = search(question, number_of_results)
    docs = list(results["texts"])

    # ----- 2. Grounded generation -----
    context = "\n".join(f"- {d}" for d in docs)
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    if verbose:
        print("\n--- Prompt sent to the LLM ---\n" + prompt + "\n" + "-" * 30)

    messages = [{"role": "user", "content": prompt}]
    output = generator(messages, max_new_tokens=200, do_sample=False)
    return output[0]["generated_text"][-1]["content"]

In [ ]:
answer = rag_answer("income generated")
print("\nANSWER:", answer)

The grounded answer reports the ~$677M worldwide gross ($773M with re-releases), pulled straight from the retrieved sentence rather than from the model's parametric memory.

**Exercise 1 — probe the pipeline.**
1. Ask the RAG system three questions the Interstellar text **can** answer (e.g., release date, awards, filming locations). Check each answer against the retrieved context.
2. Ask a question the text **cannot** answer (e.g., `"What is the mass of the moon?"`). Does the model admit the information is missing, or does it hallucinate?
3. Ask the *same* questions to the bare generator (no retrieval — call `generator(...)` with just the question). What differences do you observe?
4. Modify `PROMPT_TEMPLATE` to instruct the model to reply `"I cannot answer this from the provided documents."` when the context is insufficient, and re-test question 2.

In [ ]:
# YOUR CODE HERE (Exercise 1)

**Question 1.** Define hallucination in your own words, and explain the mechanism by which RAG reduces it. Can RAG eliminate hallucination entirely? Why or why not?

## Part 2 (8.2) — Advanced RAG techniques

The basic pipeline breaks down on realistic user behavior. The lecture covered these refinements:

**1. Query rewriting.** If a question is too verbose (or refers to earlier turns of a conversation), an LLM first rewrites it into a clean query that aids the retrieval step:
> *"We have an essay due tomorrow... I could write about penguins... Let's do dolphins. Where do they live for example?"* → **`Where do dolphins live`**

**2. Query expansion (multi-query RAG).** Convert one query into multiple queries when more than one search is needed:
> *"Compare the financial results of Nvidia in 2020 vs. 2023"* → `Nvidia 2020 financial results` **and** `Nvidia 2023 financial results`

**3. Multi-hop RAG.** Some questions need *sequential* searches, where later queries depend on earlier results:
> *"Who are the largest car manufacturers in 2023? Do they each make EVs or not?"*
> - Step 1, Query 1: `largest car manufacturers 2023` → (Toyota, Volkswagen, Hyundai)
> - Step 2, Query 1: `Toyota Motor Corporation electric vehicles`
> - Step 2, Query 2: `Volkswagen AG electric vehicles`
> - Step 2, Query 3: `Hyundai Motor Company electric vehicles`

**4. Query routing.** The ability to search multiple data sources: if the model gets a question about HR, it should search the company's HR information system (e.g., Notion); if the question is about customer data, it should search the customer relationship management system (CRM, e.g., Salesforce).

**5. Reranking.** Reorders retrieved document chunks to highlight the most pertinent results first, before they are passed to the generator.

**6. Summary (context compression).** Use other, smaller LLMs to detect and remove unimportant tokens, compressing the retrieved context before it goes into the prompt.

**7. Agentic RAG.** Taken together, the LLM increasingly acts like an **agent**: it chooses *when* to retrieve, *what tools* (data sources) to use, and *how to reason* over the results. Data sources are abstracted into tools — which may support actions beyond search (e.g., *posting* to Notion, not just searching it). Only the most capable models can do this reliably.

Let's implement two of these: **query rewriting** and **reranking**.

### Query rewriting

**Exercise 2 — query rewriting.** Use the local generator to build a query rewriter, and put it in front of your RAG pipeline.

In [ ]:
REWRITE_PROMPT = """Rewrite the user's message into a short search query \
that captures the information they need. Reply with the query only.

User message: {message}
Search query:"""


def rewrite_query(message):
    # YOUR CODE HERE: prompt the generator and return its (stripped) reply
    raise NotImplementedError


def rag_with_rewriting(message, number_of_results=3):
    # YOUR CODE HERE: rewrite, then call rag_answer with the rewritten query
    raise NotImplementedError


# Test with a verbose, conversational message:
# rag_with_rewriting(
#     "hey so I'm writing a review of that space movie by Nolan, the one with "
#     "McConaughey?? my editor wants box office numbers, how much did it make"
# )

### Reranking

A reranker takes the query and the retrieved chunks, scores each chunk's relevance to the query, and **reorders them so the most pertinent results come first**. We use a small open-source reranking model:

In [ ]:
from sentence_transformers import CrossEncoder

# A small reranking model trained on the MS MARCO relevance dataset
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def rerank(query, documents, top_k=3):
    """Score every (query, document) pair and return the documents
    sorted by relevance score."""
    pairs = [(query, doc) for doc in documents]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, documents), key=lambda x: x[0], reverse=True)
    return ranked[:top_k]


# Rerank the whole archive against our query
for score, doc in rerank("how precise was the science", texts):
    print(f"{score:.3f}\t{doc}")

The reranker is much more confident about the *scientific accuracy* sentence than about any other — compare the gap between the first and second scores.

**Exercise 3 — RAG with reranking.** `rag_answer` passes the top embedding-search results straight to the LLM. Write `rag_answer_v2` that retrieves 10 candidates, **reranks** them, and passes only the top 3 to the LLM. Find (or construct) a question where v2's answer is better.

In [ ]:
def rag_answer_v2(question, num_candidates=10, top_k=3):
    # YOUR CODE HERE: retrieve -> rerank -> grounded generation
    raise NotImplementedError

**Question 2.** For each scenario, name the most appropriate technique (query rewriting / query expansion / multi-hop / query routing / agentic RAG) and justify in one sentence:
1. "What did our CEO say about remote work in the last all-hands, and does the HR policy match it?"
2. "Compare the battery capacity of the iPhone 15 and the Galaxy S24."
3. A rambling five-sentence chat message ending in a simple factual question.
4. "Find the three most-cited papers on RAG and summarize each one's evaluation methodology."

**Question 3.** The lecture noted that these techniques progressively delegate more responsibility to the LLM, until it "acts like an agent" that chooses when to retrieve, what tools to use, and how to reason. What are the risks of delegating these decisions to the LLM?

## Part 3 (8.3) — RAG evaluation

Evaluating generated answers is harder than evaluating a classifier. The paper *"Evaluating Verifiability in Generative Search Engines"* (Liu et al., 2023) uses **human evaluation** along four axes:

| Axis | Meaning |
|---|---|
| **Fluency** | Is the generated text fluent and cohesive? |
| **Perceived utility** | Is the answer helpful and informative? |
| **Citation recall** | Proportion of generated statements about the external world that are fully supported by their citations |
| **Citation precision** | Proportion of generated citations that support their associated statements |

Human evaluation is preferred but expensive, so **LLM-as-a-judge** approaches automate it: a capable LLM scores generations along these axes. The **Ragas** library does this, adding metrics such as **faithfulness** (is the answer consistent with the provided context?) and **answer relevance** (is the answer relevant to the question?).

**Question 4.** A RAG system can have perfect citation precision and terrible citation recall. Describe what its answers would look like.

**Exercise 4 — LLM-as-a-judge.** Build a simple judge using the local generator: given a question, the retrieved context, and the RAG answer, prompt the model to score the answer's **faithfulness** to the context from 1–5 and explain its score. Run it on your answers from Exercise 1 — including the question the archive could not answer. Do you trust the judge's scores? What are the limitations of a 0.5B-parameter judge?

In [ ]:
JUDGE_PROMPT = """You are evaluating a question-answering system.

Question: {question}

Context provided to the system:
{context}

System's answer: {answer}

Score the answer's faithfulness to the context on a scale of 1-5, where
5 means every claim in the answer is supported by the context and 1 means
the answer is mostly unsupported. Reply with the score and one sentence of
justification."""


def judge_faithfulness(question, context, answer):
    # YOUR CODE HERE
    raise NotImplementedError

## Part 4 — Does a bigger model help?

Our generator so far has been a 0.5-billion-parameter model, and its weaknesses kept showing: it embellished answers, hallucinated when the archive had no answer, produced a clumsy query rewrite, and (as a judge) waved through an unsupported answer.

Let's swap in **Qwen3-4B** — the same model family, 8× larger — and re-run those exact probes. The download is ~8 GB, which typically takes a minute or two on Colab. The T4 GPU fits both models at once; if you do hit an out-of-memory error, use *Runtime → Restart session* and re-run the notebook from the top, skipping the small-model cells in Part 1.

The cell below **replaces** the global `generator`, so every function we've written (`rag_answer`, `rewrite_query`, `judge_faithfulness`, ...) automatically uses the bigger model from here on.

In [ ]:
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen3-4B-Instruct-2507",
    dtype="auto",
    device_map="auto",
)

In [ ]:
# Two probes that tripped up the small model:
print(rag_answer("What award did the film win at the Academy Awards?", verbose=False))
print()
print(rag_answer("What is the mass of the moon?", verbose=False))

**Exercise 5 — small vs. large.** Re-run your probes from Exercises 1, 2, and 4 and compare the two models. Fill in a table like this with your observations:

| Probe | 0.5B model | 4B model |
|---|---|---|
| Awards question — faithful to context? | | |
| Moon question — hallucinate or decline? | | |
| Query rewrite quality (Exercise 2 message) | | |
| Judge verdict on the unsupported moon answer | | |

For the last row: judge the *same* fabricated answer with both models, so you are comparing the judges, not the answers. If even the bigger judge gives a verdict you disagree with, read its justification closely — is the problem the model's capability, or your prompt's *definition* of faithfulness? Try tightening the rubric and re-judging.

In [ ]:
# YOUR CODE HERE (Exercise 5)

## Part 5 — Chat with your data: a Gradio RAG chatbot

In Prac 1 you built a chatbot with Gradio's `ChatInterface`, where the chat function passed the conversation straight to the model. To turn it into a **"chat with your data"** app — the RAG use case from the lecture — the only change needed is to route each message through `rag_answer` instead of the bare pipeline:

In [ ]:
import gradio as gr


def chat_response(message, history):
    # Every message is answered from the Interstellar archive via RAG.
    # (We ignore `history` here -- see the note below.)
    return rag_answer(message, verbose=False)


demo = gr.ChatInterface(
    fn=chat_response,
    title="Chat with the Interstellar archive",
    description="Ask questions about the film Interstellar — answers are grounded in the Wikipedia excerpt.",
    examples=[["When did Interstellar premiere?"], ["How accurate is the science?"]],
)

demo.launch()

Try it: ask the example questions, then ask something the archive cannot answer and watch how the model behaves.

Unlike Prac 1's chatbot, `chat_response` ignores `history` — each question is answered independently. Making the chatbot handle follow-ups like *"and who directed it?"* is exactly the **query rewriting** problem from Part 2: the message plus the history must first be rewritten into a standalone search query. If you finish early, connect your `rewrite_query` from Exercise 2 into `chat_response` and test a two-turn conversation.

## Summary

Following the Week 8 lecture, you have:

- **What is RAG (8.1):** LLMs hallucinate — fluent, syntactically correct, but factually inaccurate output. RAG reduces this with a two-step pipeline: a **search step** (embedding-based retrieval of relevant chunks) followed by **grounded generation** (the LLM is prompted with the question plus the retrieved documents).
- **Advancements in RAG (8.2):** query rewriting, query expansion (multi-query), multi-hop RAG, query routing, reranking, summary/context compression, and agentic RAG — each delegating more responsibility to the LLM, up to an agent that chooses when to retrieve, what tools to use, and how to reason.
- **RAG evaluation methods (8.3):** human evaluation along fluency, perceived utility, citation recall, and citation precision — and LLM-as-a-judge automation (e.g., Ragas: faithfulness, answer relevance).
- **Model scale (Part 4):** the same pipeline with an 8× larger generator — better faithfulness, declining instead of hallucinating, cleaner query rewrites — and a judge that can genuinely check claims against the context, provided the rubric defines faithfulness precisely.
- **Chat with your data (Part 5):** Prac 1's Gradio chatbot, grounded on an archive through RAG.

### Optional extensions

- Swap the Interstellar text for your own documents (lecture notes, a PDF's text) and rebuild the pipeline end to end.
- Implement query expansion: use the generator to produce multiple queries, retrieve for each, and pool the results before generation.
- Try Ragas to score your Exercise 1 answers for faithfulness and answer relevance.

### References

- Jay Alammar & Maarten Grootendorst, *Hands-On Large Language Models*, O'Reilly, Chapter 8.
- Alansari & Luqman (2025). *A comprehensive survey of hallucination in large language models.* arXiv:2510.06265.
- Gao et al. (2023). *Retrieval-augmented generation for large language models: A survey.* arXiv:2312.10997.
- Lewis et al. (2020). *Retrieval-augmented generation for knowledge-intensive NLP tasks.* NeurIPS 33.
- Liu, Zhang & Liang (2023). *Evaluating verifiability in generative search engines.* arXiv:2304.09848.
- Singh et al. (2025). *Agentic retrieval-augmented generation: A survey on agentic RAG.* arXiv:2501.09136.